# SimpleNN Hyperparameter Tuning

This notebook implements hyperparameter search for the SimpleNN quantum error correction model.

**Purpose:** Compare simple feedforward NN (no graph structure) against GraphSAGE to determine if graph structure helps.

**Workflow:**
1. Generate flat dataset cache (if not exists)
2. Run the training loop (trains all 50 configs sequentially)
3. Run the analysis cells to find the best configuration

**Distributed Mode (optional):**
Set `DISTRIBUTED_MODE = True` and `WORKER_ID` to distribute across multiple machines.

In [1]:
#==============================================================================
# MODE CONFIGURATION
#==============================================================================
DISTRIBUTED_MODE = False  # Set to True for multi-machine distributed training
WORKER_ID = 1  # Only used if DISTRIBUTED_MODE = True (set to 1-5)
#==============================================================================

if DISTRIBUTED_MODE:
    assert WORKER_ID in [1, 2, 3, 4, 5], f"WORKER_ID must be 1, 2, 3, 4, or 5, got {WORKER_ID}"
    if WORKER_ID == 5:
        my_config_ids = list(range(40, 50))
    else:
        my_config_ids = [i for i in range(40) if i % 4 == (WORKER_ID - 1)]
    print(f"DISTRIBUTED MODE - Worker {WORKER_ID}")
    print(f"This worker will train {len(my_config_ids)} configs: {my_config_ids}")
else:
    my_config_ids = list(range(50))  # Train all 50 configs
    print(f"LOCAL MODE - Training all 50 configs sequentially")

LOCAL MODE - Training all 50 configs sequentially


#### Imports

In [2]:
# Install required libraries (uncomment and run if needed)
# !pip install stim pymatching numpy matplotlib torch tqdm

In [ ]:
import sys
import json
import random
import time
from pathlib import Path
from datetime import datetime

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('..')

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

# Import from benchmark_models.py
from benchmark_models import (
    SurfaceCodeSampler,
    SimpleNNModel,
    SimpleNN,
    FlatDatasetCache,
)


# Set up paths
TUNING_DIR = BASE_PATH / "nn" / "tuning"
TUNING_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = TUNING_DIR / "results" / "revised_training"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = TUNING_DIR / "models" / "revised_training"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = TUNING_DIR / "plots" / "revised_training"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  TUNING_DIR: {TUNING_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  MODELS_DIR: {MODELS_DIR}")

Using device: cuda
Using device: cuda
GPU: NVIDIA GeForce GTX 1080

Paths:
  BASE_PATH: ..
  TUNING_DIR: ..\nn\tuning
  RESULTS_DIR: ..\nn\tuning\results
  MODELS_DIR: ..\nn\tuning\models


#### Generate Configs (ALREADY RUN)

Config generation - 50 configurations for hyperparameter search.
The `configs.json` file will be created on first run.

In [4]:
# # =============================================================================
# # CONFIG GENERATION - RUN ONCE ON ANY INSTANCE
# # =============================================================================

# CONFIGS_PATH = TUNING_DIR / "configs.json"

# # Check if configs already exist
# if CONFIGS_PATH.exists():
#     print(f"configs.json already exists at {CONFIGS_PATH}")
#     print("Skipping generation. Delete the file to regenerate.")
# else:
#     # Search space definition for SimpleNN
#     # Note: SimpleNN has fixed architecture (256->512->1024->1)
#     # We vary: learning rate, batch size, and add dropout variants
#     SEARCH_SPACE = {
#         'hidden_dims': [(256, 512, 1024), (128, 256, 512), (512, 1024, 2048), (64, 128, 256)],
#         'learning_rate': [1e-4, 3e-4, 5e-4, 1e-3, 3e-3],
#         'batch_size': [64, 128, 256, 512],
#         'dropout': [0.0, 0.1, 0.2, 0.3],
#     }

#     # Fixed parameters (not searched)
#     FIXED_PARAMS = {
#         'epochs': 50,
#         'distance': 7,
#     }

#     # Number of configurations to generate
#     N_CONFIGS = 50
#     SEED = 42

#     # Set seed for reproducibility
#     random.seed(SEED)

#     # Generate random configurations
#     configs = []
#     for i in range(N_CONFIGS):
#         config = {
#             'id': i,
#             'hidden_dims': random.choice(SEARCH_SPACE['hidden_dims']),
#             'learning_rate': random.choice(SEARCH_SPACE['learning_rate']),
#             'batch_size': random.choice(SEARCH_SPACE['batch_size']),
#             'dropout': random.choice(SEARCH_SPACE['dropout']),
#         }
#         configs.append(config)

#     # Create the full config document
#     config_doc = {
#         'seed': SEED,
#         'n_configs': N_CONFIGS,
#         'generated_at': datetime.now().isoformat(),
#         'search_space': {k: [str(v) if isinstance(v, tuple) else v for v in vals] for k, vals in SEARCH_SPACE.items()},
#         'fixed_params': FIXED_PARAMS,
#         'configs': configs
#     }

#     # Save to file
#     with open(CONFIGS_PATH, 'w') as f:
#         json.dump(config_doc, f, indent=2)

#     print(f"Generated {N_CONFIGS} configurations with seed {SEED}")
#     print(f"Saved to: {CONFIGS_PATH}")
#     print(f"\nSearch space:")
#     for key, values in SEARCH_SPACE.items():
#         print(f"  {key}: {values}")
#     print(f"\nFixed parameters:")
#     for key, value in FIXED_PARAMS.items():
#         print(f"  {key}: {value}")
#     print(f"\nFirst 5 configs:")
#     for c in configs[:5]:
#         print(f"  {c}")

#### Training Loop

This is the main training cell. It will:
1. Load the configs assigned to this worker
2. Skip any configs already completed (for resume support)
3. Train each config and save results incrementally

In [5]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_configs():
    """Load all configurations from configs.json."""
    configs_path = TUNING_DIR / "configs.json"
    if not configs_path.exists():
        raise FileNotFoundError(f"configs.json not found at {configs_path}. Run the config generation cell first.")
    with open(configs_path, 'r') as f:
        return json.load(f)

def get_my_configs(all_configs, config_ids):
    """Get configs by ID list."""
    return [c for c in all_configs['configs'] if c['id'] in config_ids]

def get_completed_ids():
    """Get set of already-completed config IDs from results file."""
    results_path = RESULTS_DIR / "results.json"
    if not results_path.exists():
        return set()
    with open(results_path, 'r') as f:
        results = json.load(f)
    return {r['config_id'] for r in results if r.get('status') == 'completed'}

def save_result(result):
    """Append a result to the results file."""
    results_path = RESULTS_DIR / "results.json"

    if results_path.exists():
        with open(results_path, 'r') as f:
            results = json.load(f)
    else:
        results = []

    results.append(result)

    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    return len(results)

def evaluate_model(model, detections, labels, device, batch_size=256):
    """Evaluate model accuracy on a set of samples."""
    model.model.eval()

    correct = 0
    total = len(labels)

    with torch.no_grad():
        for i in range(0, total, batch_size):
            X = detections[i:i+batch_size]
            y = labels[i:i+batch_size]
            pred = model.model(X)
            correct += ((pred.squeeze() > 0.5).float() == y).sum().item()

    return correct / total if total > 0 else 0.0

def load_and_split_dataset(cache_name, train_ratio=0.8, val_ratio=0.1, seed=42):
    """Load flat dataset from cache and split into train/val/test."""
    print(f"Loading flat dataset: {cache_name}")
    cache = FlatDatasetCache(base_path=BASE_PATH, device=device)
    cache.load(cache_name, verbose=True)

    detections, labels = cache.get_data()
    n = len(labels)

    # Shuffle with fixed seed
    torch.manual_seed(seed)
    perm = torch.randperm(n, device=device)
    detections = detections[perm]
    labels = labels[perm]

    # Calculate split points
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_det = detections[:train_end]
    train_lab = labels[:train_end]
    val_det = detections[train_end:val_end]
    val_lab = labels[train_end:val_end]
    test_det = detections[val_end:]
    test_lab = labels[val_end:]

    print(f"Dataset split: {len(train_lab)} train, {len(val_lab)} val, {len(test_lab)} test")

    return (train_det, train_lab), (val_det, val_lab), (test_det, test_lab)

In [6]:
# =============================================================================
# MODEL IMPORT
# =============================================================================
# SimpleNNModel now supports configurable hidden_dims and dropout
# Parameters: SimpleNNModel(in_channels, hidden_dims=(256, 512, 1024), dropout=0.0)
# This is imported from benchmark_models.py

In [7]:
# =============================================================================
# LOAD CONFIGS AND DATASET
# =============================================================================

import gc

# Load all configurations
all_configs = load_configs()
fixed_params = all_configs['fixed_params']

# Get configs to train (based on mode setting from above)
my_configs = get_my_configs(all_configs, my_config_ids)
print(f"\nConfigs to train: {len(my_configs)}")
print(f"Config IDs: {[c['id'] for c in my_configs]}")

# Check which are already completed
completed_ids = get_completed_ids()
remaining_configs = [c for c in my_configs if c['id'] not in completed_ids]
print(f"\nAlready completed: {len(completed_ids)} configs")
print(f"Remaining: {len(remaining_configs)} configs")

if len(remaining_configs) == 0:
    print("\nAll configs are already completed!")
else:
    # Load dataset once (shared across all configs)
    cache_name = f"d{fixed_params['distance']}_baseline"
    (train_det, train_lab), (val_det, val_lab), (test_det, test_lab) = load_and_split_dataset(cache_name)
    num_detectors = train_det.shape[1]


Configs to train: 50
Config IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]

Already completed: 0 configs
Remaining: 50 configs
Loading flat dataset: d7_baseline
Loading flat dataset 'd7_baseline' (1289.4 MB)...
  Loaded 1,000,000 samples
  Detections shape: torch.Size([1000000, 336])
  Distance: d=7
  Error rates: [0.001, 0.003, 0.005, 0.007]
Dataset split: 800000 train, 100000 val, 100000 test


In [8]:
if len(remaining_configs) > 0:
    print(f"\n{'='*60}")
    print(f"Starting training - {len(remaining_configs)} configs")
    print(f"{'='*60}")

    for i, config in enumerate(remaining_configs):
        config_id = config['id']
        print(f"\n{'='*60}")
        print(f"Config {config_id} ({i+1}/{len(remaining_configs)})")
        print(f"{'='*60}")
        print(f"Parameters:")
        print(f"  hidden_dims: {config['hidden_dims']}")
        print(f"  learning_rate: {config['learning_rate']}")
        print(f"  batch_size: {config['batch_size']}")
        print(f"  dropout: {config['dropout']}")

        start_time = time.time()

        try:
            # Initialize model with config hyperparameters
            hidden_dims = tuple(config['hidden_dims']) if isinstance(config['hidden_dims'], list) else config['hidden_dims']

            model = SimpleNN(
                nickname=f"config_{config_id}",
                in_channels=num_detectors,
                hidden_dims=hidden_dims,
                dropout=config['dropout'],
                device=device,
                base_path=BASE_PATH,
                seed=config_id  # Use config_id as seed for reproducibility
            )

            # Train the model
            epoch_losses = model.train_from_data(
                detections=train_det,
                labels=train_lab,
                epochs=fixed_params['epochs'],
                batch_size=config['batch_size'],
                lr=config['learning_rate'],
                verbose=True
            )

            # Evaluate on validation and test sets
            val_accuracy = evaluate_model(model, val_det, val_lab, device)
            test_accuracy = evaluate_model(model, test_det, test_lab, device)

            training_time = time.time() - start_time

            # Save the model to tuning/models/
            model_path = MODELS_DIR / f"config_{config_id}.pt"
            save_dict = {
                'state_dict': model.model.state_dict(),
                'config': model._config,
                'hyperparams': config,
                'val_accuracy': val_accuracy,
                'test_accuracy': test_accuracy,
                'training_time_sec': training_time,
                'timestamp': datetime.now().isoformat()
            }
            torch.save(save_dict, model_path)
            print(f"\nModel saved to: {model_path}")

            # Create result record
            result = {
                'config_id': config_id,
                'config': config,
                'val_accuracy': val_accuracy,
                'test_accuracy': test_accuracy,
                'training_time_sec': training_time,
                'final_loss': epoch_losses[-1] if epoch_losses else None,
                'model_path': str(model_path),
                'status': 'completed',
                'timestamp': datetime.now().isoformat()
            }

            print(f"\nResults:")
            print(f"  Val Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")
            print(f"  Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
            print(f"  Training Time: {training_time:.1f}s ({training_time/60:.1f} min)")

        except Exception as e:
            training_time = time.time() - start_time
            print(f"\nERROR: Config {config_id} failed: {str(e)}")
            import traceback
            traceback.print_exc()
            result = {
                'config_id': config_id,
                'config': config,
                'val_accuracy': None,
                'test_accuracy': None,
                'training_time_sec': training_time,
                'final_loss': None,
                'model_path': None,
                'status': 'failed',
                'error': str(e),
                'timestamp': datetime.now().isoformat()
            }
            # Cleanup on error too
            if 'model' in dir():
                del model

        # Save result incrementally
        n_saved = save_result(result)
        print(f"Result saved ({n_saved} total)")

        # Aggressive cleanup to prevent RAM/VRAM from filling up
        if 'model' in dir():
            del model
        if 'epoch_losses' in dir():
            del epoch_losses
        if 'save_dict' in dir():
            del save_dict

        # Force garbage collection
        gc.collect()

        # Clear CUDA cache if available
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        print(f"Memory cleared.")

    print(f"\n{'='*60}")
    print(f"TUNING COMPLETE!")
    print(f"{'='*60}")
    print(f"Total configs trained: {len(remaining_configs)}")
    print(f"Results saved to: {RESULTS_DIR / 'results.json'}")
    print(f"Models saved to: {MODELS_DIR}")


Starting training - 50 configs

Config 0 (1/50)
Parameters:
  hidden_dims: [256, 512, 1024]
  learning_rate: 0.0001
  batch_size: 256
  dropout: 0.1
SimpleNN initialized: SimpleNN(nickname='config_0', in_channels=336, hidden_dims=(256, 512, 1024), dropout=0.1)

Training: SimpleNN(nickname='config_0', in_channels=336, hidden_dims=(256, 512, 1024), dropout=0.1)
Epochs: 10 | Batch size: 256 | LR: 0.0001
Training samples: 800000


Epoch 10/10: 100%|██████████| 3125/3125 [00:12<00:00, 251.57it/s, loss=0.4100, acc=0.7568]



Training complete! Final loss: 0.4368

Model saved to: ..\nn\tuning\models\config_0.pt

Results:
  Val Accuracy: 0.7627 (76.27%)
  Test Accuracy: 0.7612 (76.12%)
  Training Time: 146.1s (2.4 min)
Result saved (1 total)
Memory cleared.

Config 1 (2/50)
Parameters:
  hidden_dims: [128, 256, 512]
  learning_rate: 0.0003
  batch_size: 64
  dropout: 0.0
SimpleNN initialized: SimpleNN(nickname='config_1', in_channels=336, hidden_dims=(128, 256, 512), dropout=0.0)

Training: SimpleNN(nickname='config_1', in_channels=336, hidden_dims=(128, 256, 512), dropout=0.0)
Epochs: 10 | Batch size: 64 | LR: 0.0003
Training samples: 800000


Epoch 10/10: 100%|██████████| 12500/12500 [00:44<00:00, 281.78it/s, loss=0.3029, acc=0.8804]



Training complete! Final loss: 0.2553

Model saved to: ..\nn\tuning\models\config_1.pt

Results:
  Val Accuracy: 0.8478 (84.78%)
  Test Accuracy: 0.8468 (84.68%)
  Training Time: 439.6s (7.3 min)
Result saved (2 total)
Memory cleared.

Config 2 (3/50)
Parameters:
  hidden_dims: [64, 128, 256]
  learning_rate: 0.0001
  batch_size: 64
  dropout: 0.0
SimpleNN initialized: SimpleNN(nickname='config_2', in_channels=336, hidden_dims=(64, 128, 256), dropout=0.0)

Training: SimpleNN(nickname='config_2', in_channels=336, hidden_dims=(64, 128, 256), dropout=0.0)
Epochs: 10 | Batch size: 64 | LR: 0.0001
Training samples: 800000


...


Training complete! Final loss: 0.3563

Model saved to: ..\nn\tuning\models\config_47.pt

Results:
  Val Accuracy: 0.8200 (82.00%)
  Test Accuracy: 0.8171 (81.71%)
  Training Time: 658.5s (11.0 min)
Result saved (48 total)
Memory cleared.

Config 48 (49/50)
Parameters:
  hidden_dims: [256, 512, 1024]
  learning_rate: 0.001
  batch_size: 256
  dropout: 0.0
SimpleNN initialized: SimpleNN(nickname='config_48', in_channels=336, hidden_dims=(256, 512, 1024), dropout=0.0)

Training: SimpleNN(nickname='config_48', in_channels=336, hidden_dims=(256, 512, 1024), dropout=0.0)
Epochs: 10 | Batch size: 256 | LR: 0.001
Training samples: 800000


Epoch 10/10: 100%|██████████| 3125/3125 [00:20<00:00, 156.20it/s, loss=0.1967, acc=0.8780]



Training complete! Final loss: 0.2633

Model saved to: ..\nn\tuning\models\config_48.pt

Results:
  Val Accuracy: 0.8660 (86.60%)
  Test Accuracy: 0.8644 (86.44%)
  Training Time: 196.9s (3.3 min)
Result saved (49 total)
Memory cleared.

Config 49 (50/50)
Parameters:
  hidden_dims: [128, 256, 512]
  learning_rate: 0.0003
  batch_size: 128
  dropout: 0.3
SimpleNN initialized: SimpleNN(nickname='config_49', in_channels=336, hidden_dims=(128, 256, 512), dropout=0.3)

Training: SimpleNN(nickname='config_49', in_channels=336, hidden_dims=(128, 256, 512), dropout=0.3)
Epochs: 10 | Batch size: 128 | LR: 0.0003
Training samples: 800000


Epoch 10/10: 100%|██████████| 6250/6250 [00:33<00:00, 187.65it/s, loss=0.3100, acc=0.8182]



Training complete! Final loss: 0.3673

Model saved to: ..\nn\tuning\models\config_49.pt

Results:
  Val Accuracy: 0.8240 (82.40%)
  Test Accuracy: 0.8245 (82.45%)
  Training Time: 288.7s (4.8 min)
Result saved (50 total)
Memory cleared.

TUNING COMPLETE!
Total configs trained: 50
Results saved to: ..\nn\tuning\results\results.json
Models saved to: ..\nn\tuning\models


#### Analyze Results

This cell loads results (from local `results.json` or distributed worker files) and identifies the best configurations.

In [9]:
# =============================================================================
# LOAD ALL RESULTS
# =============================================================================

def load_all_results():
    """Load results from local mode or distributed worker files."""
    all_results = []

    # First check for local results.json (local mode)
    local_results_path = RESULTS_DIR / "results.json"
    if local_results_path.exists():
        with open(local_results_path, 'r') as f:
            local_results = json.load(f)
        all_results.extend(local_results)
        print(f"Local results: {len(local_results)} configs")

    # Also check for distributed worker files
    for worker_id in [1, 2, 3, 4, 5]:
        results_path = RESULTS_DIR / f"worker_{worker_id}.json"
        if results_path.exists():
            with open(results_path, 'r') as f:
                worker_results = json.load(f)
            all_results.extend(worker_results)
            print(f"Worker {worker_id}: {len(worker_results)} results")

    # Deduplicate by config_id (in case of overlap)
    seen_ids = set()
    unique_results = []
    for r in all_results:
        if r['config_id'] not in seen_ids:
            seen_ids.add(r['config_id'])
            unique_results.append(r)

    if len(unique_results) < len(all_results):
        print(f"\n(Removed {len(all_results) - len(unique_results)} duplicate entries)")

    return unique_results

# Load results
print("Loading results...")
all_results = load_all_results()

# Filter to completed only
completed_results = [r for r in all_results if r.get('status') == 'completed']
failed_results = [r for r in all_results if r.get('status') == 'failed']

print(f"\nTotal results: {len(all_results)}")
print(f"  Completed: {len(completed_results)}")
print(f"  Failed: {len(failed_results)}")

if len(completed_results) > 0:
    # Sort by test accuracy
    sorted_results = sorted(completed_results, key=lambda x: x['test_accuracy'], reverse=True)

    print(f"\n{'='*60}")
    print(f"TOP 10 CONFIGURATIONS")
    print(f"{'='*60}")

    for i, r in enumerate(sorted_results[:10]):
        print(f"\n{i+1}. Config {r['config_id']}")
        print(f"   Test Accuracy: {r['test_accuracy']:.4f} ({r['test_accuracy']*100:.2f}%)")
        print(f"   Val Accuracy: {r['val_accuracy']:.4f}")
        print(f"   Hidden dims: {r['config']['hidden_dims']}")
        print(f"   LR: {r['config']['learning_rate']}, Batch: {r['config']['batch_size']}, Dropout: {r['config']['dropout']}")
        print(f"   Training time: {r['training_time_sec']/60:.1f} min")

Loading results...
Local results: 50 configs

Total results: 50
  Completed: 50
  Failed: 0

TOP 10 CONFIGURATIONS

1. Config 33
   Test Accuracy: 0.8776 (87.76%)
   Val Accuracy: 0.8769
   Hidden dims: [512, 1024, 2048]
   LR: 0.0003, Batch: 64, Dropout: 0.1
   Training time: 9.0 min

2. Config 30
   Test Accuracy: 0.8713 (87.13%)
   Val Accuracy: 0.8720
   Hidden dims: [256, 512, 1024]
   LR: 0.0005, Batch: 128, Dropout: 0.1
   Training time: 7.4 min

3. Config 3
   Test Accuracy: 0.8684 (86.84%)
   Val Accuracy: 0.8682
   Hidden dims: [128, 256, 512]
   LR: 0.0003, Batch: 64, Dropout: 0.1
   Training time: 8.4 min

4. Config 21
   Test Accuracy: 0.8683 (86.83%)
   Val Accuracy: 0.8672
   Hidden dims: [512, 1024, 2048]
   LR: 0.0003, Batch: 128, Dropout: 0.2
   Training time: 5.0 min

5. Config 31
   Test Accuracy: 0.8675 (86.75%)
   Val Accuracy: 0.8676
   Hidden dims: [512, 1024, 2048]
   LR: 0.0003, Batch: 64, Dropout: 0.2
   Training time: 10.1 min

6. Config 8
   Test Accuracy: 

In [10]:
# =============================================================================
# SAVE COMBINED RESULTS TO CSV
# =============================================================================

if len(completed_results) > 0:
    # Create DataFrame with flattened config
    rows = []
    for r in completed_results:
        row = {
            'config_id': r['config_id'],
            'test_accuracy': r['test_accuracy'],
            'val_accuracy': r['val_accuracy'],
            'final_loss': r['final_loss'],
            'training_time_sec': r['training_time_sec'],
            'hidden_dims': str(r['config']['hidden_dims']),
            'learning_rate': r['config']['learning_rate'],
            'batch_size': r['config']['batch_size'],
            'dropout': r['config']['dropout'],
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    df = df.sort_values('test_accuracy', ascending=False)

    # Save to CSV
    csv_path = TUNING_DIR / "results_combined.csv"
    df.to_csv(csv_path, index=False)
    print(f"Results saved to: {csv_path}")

    # Display summary stats
    print(f"\nSummary Statistics:")
    print(f"  Best test accuracy: {df['test_accuracy'].max():.4f}")
    print(f"  Worst test accuracy: {df['test_accuracy'].min():.4f}")
    print(f"  Mean test accuracy: {df['test_accuracy'].mean():.4f}")
    print(f"  Std test accuracy: {df['test_accuracy'].std():.4f}")

Results saved to: ..\nn\tuning\results_combined.csv

Summary Statistics:
  Best test accuracy: 0.8776
  Worst test accuracy: 0.2793
  Mean test accuracy: 0.7684
  Std test accuracy: 0.1517
